# Pandas avanzado con gráficos

Este notebook integra operaciones avanzadas de **pandas** con visualización. Trabajaremos con un conjunto sintético y reproducible de ventas diarias. La idea no es hacer un catálogo de gráficos, sino entender el flujo:

1. **pandas** prepara, agrupa y reorganiza los datos.
2. `DataFrame.plot` y `Series.plot` ofrecen una interfaz de alto nivel.
3. **Matplotlib** crea y renderiza las figuras y los ejes (`Axes`).

Solo se utilizan pandas, NumPy y Matplotlib.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Semilla fija: cada ejecución genera los mismos datos
rng = np.random.default_rng(42)
fechas = pd.date_range("2025-01-01", periods=180, freq="D")
productos = np.array(["Curso", "Libro", "Licencia"])
regiones = np.array(["Norte", "Centro", "Sur"])
df = pd.DataFrame({
    "fecha": fechas,
    "producto": rng.choice(productos, len(fechas)),
    "region": rng.choice(regiones, len(fechas)),
    "unidades": rng.poisson(18, len(fechas)) + 1,
    "precio": rng.choice([15, 25, 40, 60], len(fechas)),
})
df["ingresos"] = df["unidades"] * df["precio"]
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").set_index("fecha")
df.head()

## 1. Preparar y enriquecer antes de dibujar

Las gráficas son más legibles cuando las variables ya expresan la pregunta. Aquí enlazamos varias operaciones avanzadas: `merge` añade un objetivo regional, `cut` crea tramos categóricos y `transform` calcula el porcentaje de cada fila dentro de su producto.

In [ ]:
objetivos = pd.DataFrame({"region": regiones, "objetivo_diario": [900, 1100, 750]})
df = df.reset_index().merge(objetivos, on="region", validate="many_to_one").set_index("fecha").sort_index()
df["tramo_unidades"] = pd.cut(
    df["unidades"], bins=[0, 15, 25, np.inf], labels=["bajo", "medio", "alto"]
)
df["cuota_producto"] = df["ingresos"] / df.groupby("producto")["ingresos"].transform("sum")
df[["producto", "region", "ingresos", "tramo_unidades", "cuota_producto"]].head()

## 2. `groupby`/`agg` y gráfico de barras

Primero resumimos: pandas hace la agregación y `DataFrame.plot(kind="bar")` la representa. Las barras permiten comparar claramente los ingresos acumulados por producto y región.

In [ ]:
resumen = (df.groupby(["producto", "region"], observed=True)
             .agg(ingresos=("ingresos", "sum"), unidades=("unidades", "sum"))
             .reset_index())
barra = resumen.pivot(index="producto", columns="region", values="ingresos").plot(
    kind="bar", figsize=(9, 4), rot=0, alpha=0.85, grid=True,
    title="Ingresos acumulados por producto y región"
)
barra.set_xlabel("Producto")
barra.set_ylabel("Ingresos (€)")
barra.legend(title="Región")
plt.tight_layout()
barra

## 3. Series temporales: `rolling` y `resample`

`rolling` suaviza la variación diaria con una ventana móvil; `resample` cambia la frecuencia temporal. La serie diaria y su media móvil se combinan sobre el mismo `Axes` mediante `ax=`.

In [ ]:
diario = df["ingresos"].resample("D").sum()
semanal = diario.resample("W").sum()
media_movil = diario.rolling(14, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(10, 4))
diario.plot(ax=ax, alpha=0.25, color="tab:blue", label="Diario")
media_movil.plot(ax=ax, color="tab:blue", linewidth=2, label="Media móvil (14 días)")
semanal.plot(ax=ax, color="tab:orange", marker="o", label="Total semanal")
ax.set_title("Evolución de ingresos: detalle, tendencia y resumen")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ingresos (€)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
ax

## 4. Histograma y dispersión

Un histograma responde cómo se distribuyen las unidades vendidas; una dispersión relaciona unidades e ingresos. `Series.plot` es apropiado cuando dibujamos una sola serie, mientras que `DataFrame.plot` facilita seleccionar columnas y colorear por una variable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["unidades"].plot(kind="hist", bins=12, ax=axes[0], alpha=0.75, color="tab:green", grid=True)
axes[0].set_title("Distribución de unidades por día")
axes[0].set_xlabel("Unidades")
axes[0].set_ylabel("Frecuencia")

df.plot(kind="scatter", x="unidades", y="ingresos", ax=axes[1], alpha=0.45, color="tab:purple", grid=True)
axes[1].set_title("Unidades e ingresos")
axes[1].set_xlabel("Unidades")
axes[1].set_ylabel("Ingresos (€)")
fig.tight_layout()
axes[0], axes[1]

## 5. Boxplot: comparar grupos

El boxplot resume mediana, cuartiles y posibles valores extremos. Es útil para no confundir un promedio alto con una distribución homogénea entre productos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column="ingresos", by="producto", ax=ax, grid=True)
ax.set_title("Variabilidad de ingresos diarios por producto")
ax.set_xlabel("Producto")
ax.set_ylabel("Ingresos (€)")
fig.suptitle("")
fig.tight_layout()
ax

## 6. Área: composición acumulada

Una tabla dinámica (`pivot_table`) reorganiza el índice para que cada región sea una columna. El área apilada ayuda a observar simultáneamente el total y la contribución de cada región; internamente sigue siendo una operación de pandas antes de dibujar.

In [ ]:
por_region = df.pivot_table(index="fecha", columns="region", values="ingresos", aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
por_region.plot.area(ax=ax, alpha=0.65, stacked=True, figsize=(10, 4), grid=True, title="Composición diaria de ingresos por región")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ingresos (€)")
ax.legend(title="Región", loc="upper left")
fig.tight_layout()
ax

## 7. MultiIndex y lectura docente

El resumen conserva dos niveles (`producto`, `region`) antes de reorganizarlo con `pivot`. Un gráfico no sustituye al análisis:

- las barras comparan magnitudes acumuladas, pero pueden ocultar la evolución temporal;
- la línea muestra tendencia y estacionalidad, y la media móvil reduce ruido;
- el histograma describe frecuencias, la dispersión permite buscar relación entre variables y el boxplot revela variabilidad;
- el área comunica composición, pero las regiones posteriores quedan visualmente condicionadas por las anteriores.

Elegir el gráfico depende de la pregunta y de la transformación previa.

In [ ]:
multi = resumen.set_index(["producto", "region"]).sort_index()
print("Índice del resumen:", type(multi.index).__name__)
print(multi.head(3))

# Cada método plot devuelve un Axes de Matplotlib; no es un DataFrame nuevo.
fig, ax = plt.subplots(figsize=(7, 3))
objeto_devuelto = semanal.plot(ax=ax, kind="line", marker="o", title="Comprobación de Series.plot")
ax.set_xlabel("Semana")
ax.set_ylabel("Ingresos (€)")
ax.grid(True)
fig.tight_layout()
print("¿Es un matplotlib.axes.Axes?:", isinstance(objeto_devuelto, plt.Axes))
objeto_devuelto

## 8. Guardado opcional sin dejar basura

En una práctica real se puede guardar una figura como PNG. Esta demostración usa una carpeta temporal y elimina el archivo al terminar; así el notebook sigue siendo autocontenido y no contamina el repositorio.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="pandas-graficos-") as carpeta: 
    destino = Path(carpeta) / "ingresos_semanales.png"
    fig.savefig(destino, dpi=120, bbox_inches="tight")
    print(f"Figura guardada opcionalmente en: {destino.name} ({destino.stat().st_size} bytes)")
print("La carpeta temporal se elimina al salir del bloque.")